# Real Pendulum Reference Tracking

This notebook runs and optionally fine-tunes a reference-tracking policy on the physical unbalanced disk. The policy observation is five-dimensional:

`[sin(theta), cos(theta), omega_corrected, sin(theta_ref), cos(theta_ref)]`

Use the omega scale/offset found in `omega_calibration.ipynb`. The correction is applied at the hardware interface; the saved policy itself does not contain the correction.

## Recommended Workflow

1. Calibrate omega in `omega_calibration.ipynb`.
2. Paste `OMEGA_SCALE` and `OMEGA_OFFSET` below.
3. Deploy the sim-trained reference policy with corrected omega, without learning.
4. Only run real augmentation if corrected deployment is not good enough.

For deployment, choose one fixed target offset such as `TARGET_DEG = 0`, `5`, `10`, or `15`. For augmentation, the wrapper samples references in `[-REF_RANGE_DEG, +REF_RANGE_DEG]` at every reset, matching the simulation reference-tracking setup.

## 1. Setup

In [ ]:
from pathlib import Path
import importlib
import sys
import time

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import usb.core
from stable_baselines3 import A2C, PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / 'gym_unbalanced_disk').exists():
        sys.path.insert(0, str(parent))
        break

import gym_unbalanced_disk
import a2c_reference_tracking_train as ref_rl
importlib.reload(ref_rl)

DT = 0.025
UMAX = 3.0
ALGORITHM = ref_rl.ALGORITHM  # 'PPO' or 'A2C'; must match the .zip policy
REF_RANGE_DEG = ref_rl.REF_RANGE_DEG
REF_RANGE_RAD = np.deg2rad(REF_RANGE_DEG)

# Paste values from omega_calibration.ipynb here.
OMEGA_SCALE = 1.0
OMEGA_OFFSET = 0.0

# Optional low-pass filtering for real omega. 0.0 disables filtering.
# Use only mild filtering; too much filtering adds delay.
OMEGA_FILTER_ALPHA = 0.0

REAL_EPISODE_STEPS = 300
REAL_SIMULATION_STEPS = 1200
MAX_ABS_OMEGA_CORRECTED = 35.0
USB_RESET_RETRIES = 3
USB_RETRY_SLEEP = 0.25
USB_TIMEOUT_PENALTY = -5.0

ModelClass = PPO if ALGORITHM.upper() == 'PPO' else A2C

print(f'Algorithm:        {ALGORITHM}')
print(f'Reference range:  +/-{REF_RANGE_DEG} deg')
print(f'Omega correction: omega_corrected = {OMEGA_SCALE:+.6f} * omega_raw + {OMEGA_OFFSET:+.6f}')
print(f'Omega filter:     alpha={OMEGA_FILTER_ALPHA}')

## 2. Observation Helpers

In [ ]:
def theta_ref_from_offset_deg(offset_deg):
    return np.pi + np.deg2rad(float(offset_deg))


def append_reference_obs(base_obs, theta_ref):
    ref_obs = np.array([np.sin(theta_ref), np.cos(theta_ref)], dtype=np.float32)
    return np.concatenate([np.asarray(base_obs, dtype=np.float32), ref_obs]).astype(np.float32)


def corrected_base_obs(raw_obs, previous_omega=None):
    obs = np.asarray(raw_obs, dtype=np.float32).copy()
    omega_corrected = OMEGA_SCALE * float(obs[2]) + OMEGA_OFFSET
    if previous_omega is not None and OMEGA_FILTER_ALPHA > 0.0:
        omega_corrected = OMEGA_FILTER_ALPHA * previous_omega + (1.0 - OMEGA_FILTER_ALPHA) * omega_corrected
    obs[2] = omega_corrected
    return obs, float(omega_corrected)


def make_policy_obs(raw_obs, theta_ref, previous_omega=None):
    base_obs, omega_corrected = corrected_base_obs(raw_obs, previous_omega)
    return append_reference_obs(base_obs, theta_ref), omega_corrected

## 3. Deploy/Test a Reference Policy with Corrected Omega

Run this before augmentation. It tests a fixed target reference and does not learn.

In [ ]:
RUN_REFERENCE_DEPLOYMENT = False

# Point this to a 5D reference-tracking policy, not a 3D swing-up policy.
DEPLOY_MODEL_ZIP = Path('ppo_ref_track_v2.zip')
TARGET_DEG = 0.0
DEPLOY_STEPS = REAL_SIMULATION_STEPS
DEPLOY_DETERMINISTIC = True
ACTION_SIGN = 1.0
PRINT_FIRST_STEPS = 20
SAVE_DEPLOYMENT_LOG = True
DEPLOY_LOG_DIR = Path('real_reference_tracking_logs')
DEPLOY_LOG_DIR.mkdir(exist_ok=True)

deploy_candidates = [
    DEPLOY_MODEL_ZIP,
    Path.cwd() / DEPLOY_MODEL_ZIP,
    Path.cwd() / 'gym-unbalanced-disk-master' / 'gym_unbalanced_disk' / 'examples-connect-to-exp' / DEPLOY_MODEL_ZIP,
    Path.cwd() / 'ML-for-control-systems-Pendulum' / 'gym-unbalanced-disk-master' / 'gym_unbalanced_disk' / 'examples-connect-to-exp' / DEPLOY_MODEL_ZIP,
]
deploy_model_path = next((path for path in deploy_candidates if path.exists()), deploy_candidates[0])

print(f'Deploy model: {deploy_model_path}')
print(f'Target offset: {TARGET_DEG:+.1f} deg')

In [ ]:
if RUN_REFERENCE_DEPLOYMENT:
    if not deploy_model_path.exists():
        raise FileNotFoundError(f'Could not find DEPLOY_MODEL_ZIP. Checked: {deploy_candidates}')

    deploy_model = ModelClass.load(deploy_model_path, device='cpu')
    deploy_env = gym_unbalanced_disk.UnbalancedDisk_exp_sincos(dt=DT, umax=UMAX)
    theta_ref = theta_ref_from_offset_deg(TARGET_DEG)

    policy_obs_list = []
    raw_obs_list = []
    theta_list = []
    theta_ref_list = []
    ref_error_list = []
    omega_raw_list = []
    omega_corrected_list = []
    u_model_list = []
    u_applied_list = []

    previous_omega = None
    try:
        raw_obs, info = deploy_env.reset()
        for k in range(DEPLOY_STEPS):
            policy_obs, omega_corrected = make_policy_obs(raw_obs, theta_ref, previous_omega)
            previous_omega = omega_corrected
            action, _ = deploy_model.predict(policy_obs, deterministic=DEPLOY_DETERMINISTIC)
            u_model = float(np.asarray(action).reshape(-1)[0])
            u_applied = float(np.clip(ACTION_SIGN * u_model, -UMAX, UMAX))

            theta = float(deploy_env.th)
            ref_error = abs(ref_rl.wrap_angle(theta - theta_ref))

            raw_obs_list.append(np.asarray(raw_obs, dtype=np.float32).copy())
            policy_obs_list.append(policy_obs.copy())
            theta_list.append(theta)
            theta_ref_list.append(theta_ref)
            ref_error_list.append(ref_error)
            omega_raw_list.append(float(raw_obs[2]))
            omega_corrected_list.append(omega_corrected)
            u_model_list.append(u_model)
            u_applied_list.append(u_applied)

            if k < PRINT_FIRST_STEPS:
                print(
                    f'k={k:03d} policy_obs={np.round(policy_obs, 3)} '
                    f'u_model={u_model:+.3f} u_applied={u_applied:+.3f}'
                )

            raw_obs, reward, terminal, truncated, info = deploy_env.step(u_applied)
            if terminal or truncated:
                break
    finally:
        try:
            deploy_env.step(0.0)
        finally:
            deploy_env.close()

    raw_obs_arr = np.asarray(raw_obs_list)
    policy_obs_arr = np.asarray(policy_obs_list)
    theta_arr = np.asarray(theta_list)
    ref_error_arr = np.asarray(ref_error_list)
    omega_raw_arr = np.asarray(omega_raw_list)
    omega_corrected_arr = np.asarray(omega_corrected_list)
    u_model_arr = np.asarray(u_model_list)
    u_applied_arr = np.asarray(u_applied_list)
    t = np.arange(len(theta_arr)) * DT

    print(f'Ran deployment for {len(theta_arr)} steps ({len(theta_arr) * DT:.1f}s)')
    if len(theta_arr):
        print(f'mean ref error:  {np.degrees(ref_error_arr.mean()):.2f} deg')
        print(f'final ref error: {np.degrees(ref_error_arr[-1]):.2f} deg')
        print(f'max |u|:         {np.max(np.abs(u_applied_arr)):.2f} V')
        print(f'max |omega|:     {np.max(np.abs(omega_corrected_arr)):.2f} rad/s')

    if SAVE_DEPLOYMENT_LOG:
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = DEPLOY_LOG_DIR / f'ref_deploy_{deploy_model_path.stem}_{TARGET_DEG:+.0f}_{stamp}.npz'
        np.savez(
            log_path,
            t=t,
            raw_obs=raw_obs_arr,
            policy_obs=policy_obs_arr,
            theta=theta_arr,
            theta_ref=np.asarray(theta_ref_list),
            ref_error=ref_error_arr,
            omega_raw=omega_raw_arr,
            omega_corrected=omega_corrected_arr,
            u_model=u_model_arr,
            u_applied=u_applied_arr,
            omega_scale=OMEGA_SCALE,
            omega_offset=OMEGA_OFFSET,
            target_deg=TARGET_DEG,
            model_path=str(deploy_model_path),
        )
        print(f'Saved deployment log: {log_path}')

    fig, axs = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
    axs[0].plot(t, np.degrees([ref_rl.wrap_angle(x) for x in theta_arr]), label='theta')
    axs[0].axhline(np.degrees(ref_rl.wrap_angle(theta_ref)), color='k', ls='--', alpha=0.5, label='theta_ref')
    axs[0].set_ylabel('angle [deg]')
    axs[0].legend()
    axs[1].plot(t, np.degrees(ref_error_arr))
    axs[1].set_ylabel('|error| [deg]')
    axs[2].plot(t, omega_raw_arr, label='raw')
    axs[2].plot(t, omega_corrected_arr, label='corrected')
    axs[2].set_ylabel('omega [rad/s]')
    axs[2].legend()
    axs[3].plot(t, u_model_arr, label='model')
    axs[3].plot(t, u_applied_arr, label='applied', ls='--')
    axs[3].set_ylabel('u [V]')
    axs[3].set_xlabel('time [s]')
    axs[3].set_ylim(-UMAX - 0.2, UMAX + 0.2)
    axs[3].legend()
    for ax in axs:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print('Reference deployment skipped. Set RUN_REFERENCE_DEPLOYMENT = True when connected to the lab setup.')

## 4. Real Augmentation/Fine-Tuning for Reference Tracking

Run this only after deployment with corrected omega has been tested. The real training environment samples a new reference offset in `[-REF_RANGE_DEG, +REF_RANGE_DEG]` at every reset and computes the same reference-centered reward as simulation, using corrected omega.

In [ ]:
RUN_REFERENCE_AUGMENTATION = False

# Starting .zip must be a 5D reference-tracking policy.
BASE_MODEL_ZIP = Path('ppo_ref_track_v2.zip')
RUN_NAME = 'real_reference_tracking_augmented'
REAL_TRAIN_EPISODES = 250
REAL_TRAIN_STEPS = REAL_TRAIN_EPISODES * REAL_EPISODE_STEPS
CHECKPOINT_EVERY_EPISODES = 25
REAL_LEARNING_RATE = 1e-5

base_candidates = [
    BASE_MODEL_ZIP,
    Path.cwd() / BASE_MODEL_ZIP,
    Path.cwd() / 'gym-unbalanced-disk-master' / 'gym_unbalanced_disk' / 'examples-connect-to-exp' / BASE_MODEL_ZIP,
    Path.cwd() / 'ML-for-control-systems-Pendulum' / 'gym-unbalanced-disk-master' / 'gym_unbalanced_disk' / 'examples-connect-to-exp' / BASE_MODEL_ZIP,
]
base_model_path = next((path for path in base_candidates if path.exists()), base_candidates[0])
run_dir = Path('real_reference_tracking_checkpoints') / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
augmented_model_path = run_dir / f'{base_model_path.stem}_{RUN_NAME}.zip'

print(f'Base model:      {base_model_path}')
print(f'Run dir:         {run_dir}')
print(f'Output model:    {augmented_model_path}')
print(f'Train episodes:  {REAL_TRAIN_EPISODES}')
print(f'Learning rate:   {REAL_LEARNING_RATE}')

In [ ]:
class CorrectedOmegaRealRefTrackEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self, dt=DT, umax=UMAX, max_episode_steps=REAL_EPISODE_STEPS, max_abs_omega=35.0):
        super().__init__()
        self.dt = dt
        self.umax = umax
        self.max_episode_steps = max_episode_steps
        self.max_abs_omega = max_abs_omega
        self.steps = 0
        self.theta_ref = np.pi
        self.previous_omega = None
        self.env = self._make_hardware_env()
        self.action_space = gym.spaces.Box(
            low=np.array([-umax], dtype=np.float32),
            high=np.array([umax], dtype=np.float32),
            dtype=np.float32,
        )
        self.observation_space = gym.spaces.Box(
            low=np.array([-1., -1., -40., -1., -1.], dtype=np.float32),
            high=np.array([1.,  1.,  40.,  1.,  1.], dtype=np.float32),
            dtype=np.float32,
        )
        self.last_obs = append_reference_obs(np.array([0.0, 1.0, 0.0], dtype=np.float32), self.theta_ref)

    def _make_hardware_env(self):
        return gym_unbalanced_disk.UnbalancedDisk_exp_sincos(dt=self.dt, umax=self.umax)

    def _safe_zero(self):
        try:
            self.env.step(0.0)
        except usb.core.USBError as exc:
            print(f'USB error while sending zero command: {exc}')

    def _reconnect(self):
        try:
            self.env.close()
        except Exception:
            pass
        time.sleep(USB_RETRY_SLEEP)
        self.env = self._make_hardware_env()

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)
        self.steps = 0
        self.previous_omega = None
        self.theta_ref = np.pi + np.random.uniform(-REF_RANGE_RAD, REF_RANGE_RAD)
        last_error = None
        for attempt in range(1, USB_RESET_RETRIES + 1):
            try:
                raw_obs, info = self.env.reset(seed=seed)
                obs, omega_corrected = make_policy_obs(raw_obs, self.theta_ref, self.previous_omega)
                self.previous_omega = omega_corrected
                self.last_obs = obs
                info = {**info, 'theta_ref': float(self.theta_ref), 'omega_corrected': omega_corrected}
                return obs, info
            except usb.core.USBError as exc:
                last_error = exc
                print(f'USB reset/read error on attempt {attempt}/{USB_RESET_RETRIES}: {exc}')
                self._reconnect()
        raise last_error

    def step(self, action):
        u = float(np.asarray(action, dtype=np.float32).reshape(-1)[0])
        u = float(np.clip(u, -self.umax, self.umax))
        try:
            raw_obs, env_reward, terminated, truncated, info = self.env.step(u)
        except usb.core.USBError as exc:
            self.steps += 1
            self._safe_zero()
            info = {'usb_error': str(exc), 'u': u, 'safety_truncated': True}
            print(f'USB step/read error; ending episode and continuing training: {exc}')
            return self.last_obs, USB_TIMEOUT_PENALTY, False, True, info

        self.steps += 1
        obs, omega_corrected = make_policy_obs(raw_obs, self.theta_ref, self.previous_omega)
        self.previous_omega = omega_corrected
        self.last_obs = obs

        theta = float(self.env.th)
        omega_raw = float(raw_obs[2])
        ref_error = abs(ref_rl.wrap_angle(theta - self.theta_ref))
        reward = ref_rl.external_ref_reward(theta, omega_corrected, u, self.umax, self.theta_ref)
        safety_truncated = abs(omega_corrected) > self.max_abs_omega
        time_truncated = self.steps >= self.max_episode_steps

        info = {
            **info,
            'theta': theta,
            'omega_raw': omega_raw,
            'omega': omega_corrected,
            'omega_corrected': omega_corrected,
            'theta_ref': float(self.theta_ref),
            'ref_error': float(ref_error),
            'env_reward': float(env_reward),
            'train_reward': float(reward),
            'u': u,
            'safety_truncated': bool(safety_truncated),
        }
        return obs, float(reward), bool(terminated), bool(truncated or time_truncated or safety_truncated), info

    def close(self):
        try:
            self._safe_zero()
        finally:
            self.env.close()


def make_real_ref_env():
    return Monitor(
        CorrectedOmegaRealRefTrackEnv(
            dt=DT,
            umax=UMAX,
            max_episode_steps=REAL_EPISODE_STEPS,
            max_abs_omega=MAX_ABS_OMEGA_CORRECTED,
        )
    )

In [ ]:
class EpisodeModelCheckpoint(BaseCallback):
    def __init__(self, save_path, every_episodes=25, print_every_episodes=1):
        super().__init__()
        self.save_path = Path(save_path)
        self.every_episodes = every_episodes
        self.print_every_episodes = print_every_episodes
        self.episodes = 0
        self.recent_rewards = []

    def _on_step(self):
        infos = self.locals.get('infos', [])
        dones = self.locals.get('dones', [])
        for done, info in zip(dones, infos):
            if done:
                self.episodes += 1
                episode = info.get('episode', {})
                ep_reward = float(episode.get('r', np.nan))
                ep_length = int(episode.get('l', 0))
                if np.isfinite(ep_reward):
                    self.recent_rewards.append(ep_reward)
                    self.recent_rewards = self.recent_rewards[-10:]
                mean10 = float(np.nanmean(self.recent_rewards)) if self.recent_rewards else np.nan
                ref_deg = np.degrees(ref_rl.wrap_angle(info.get('theta_ref', np.pi) - np.pi))
                final_err_deg = np.degrees(info.get('ref_error', np.nan))
                usb_error = info.get('usb_error')
                safety = info.get('safety_truncated', False)

                if self.episodes % self.print_every_episodes == 0:
                    print(
                        f'episode {self.episodes}/{REAL_TRAIN_EPISODES} | '
                        f'reward={ep_reward:.2f} | mean10={mean10:.2f} | len={ep_length} | '
                        f'ref={ref_deg:+.1f}deg | final_err={final_err_deg:.2f}deg | '
                        f'safety={safety} | usb_error={bool(usb_error)}'
                    )

                if self.episodes % self.every_episodes == 0:
                    checkpoint_path = self.save_path / f'model_{self.episodes:03d}'
                    self.model.save(checkpoint_path)
                    print(f'Saved model checkpoint: {checkpoint_path}.zip')
        return True


if RUN_REFERENCE_AUGMENTATION:
    if not base_model_path.exists():
        raise FileNotFoundError(f'Could not find BASE_MODEL_ZIP. Checked: {base_candidates}')

    train_env = DummyVecEnv([make_real_ref_env])
    model = ModelClass.load(
        base_model_path,
        env=train_env,
        device='cpu',
        custom_objects={
            'learning_rate': REAL_LEARNING_RATE,
            'lr_schedule': lambda _: REAL_LEARNING_RATE,
        },
    )

    checkpoint_callback = EpisodeModelCheckpoint(
        save_path=run_dir,
        every_episodes=CHECKPOINT_EVERY_EPISODES,
        print_every_episodes=1,
    )

    try:
        model.learn(
            total_timesteps=REAL_TRAIN_STEPS,
            reset_num_timesteps=False,
            progress_bar=False,
            callback=checkpoint_callback,
        )
        model.save(augmented_model_path)
        print(f'Saved real reference-tracking augmented model: {augmented_model_path}')
    finally:
        train_env.close()
else:
    print('Reference augmentation skipped. Set RUN_REFERENCE_AUGMENTATION = True when connected to the lab setup.')